# rs-embed Playground
<img src="https://raw.githubusercontent.com/cybergis/rs-embed/main/docs/assets/background.png" width="800" /> 

## Motivation
Remote sensing foundation models are powerful, but using them is often messy: different models expect different sensors, time windows, preprocessing, and output shapes.
`rs-embed` simplifies that by giving you one ROI-centric request pattern that stays stable while the model and workflow scale change.

## What is embedding?
An embedding is a compact numeric representation produced by a pretrained model.
For remote sensing, you can think of it as a machine-readable description of a region of interest at a chosen time that can be reused for retrieval, clustering, classification, benchmarking, or spatial analysis.

## Why use `rs-embed`
- one API across provider-backed and precomputed embeddings
- explicit `spatial`, `temporal`, `sensor`, and `output` specs
- easy progression from one ROI to batches and exportable datasets
- built-in inspection so you can check the raw patch before blaming the model

## What this notebook shows
1. set up the package and Google Earth Engine
2. define a reusable ROI and time window
3. inspect the provider patch visually
4. run one model, then vary preprocessing and output shape
5. scale to many ROIs and export a small dataset

<img src="https://raw.githubusercontent.com/cybergis/rs-embed/main/docs/assets/scale.png" width="550" /> 

> Recommended kernel: `rsembed`


Skip these if you already install the package

In [ ]:
%cd ..
%pip install -e .
%cd examples

## 0. Before You Run



In [ ]:
import ee
ee.Authenticate()
ee.Initialize()

In [ ]:
from rs_embed import BBox, PointBuffer, TemporalSpec, OutputSpec, SensorSpec
from rs_embed import inspect_gee_patch, get_embedding, get_embeddings_batch, export_batch, inspect_provider_patch
from plot_utils import *

## 1. Define the request once
In `rs-embed`, most work starts by describing the ROI and time window, not by jumping straight to a model.
That separation is one of the package's main strengths: the same `spatial` and `temporal` specs can be reused for inspection, single-model inference, batching, and export.


In [ ]:
# Spatial: point + buffer
spatial_point = PointBuffer(
    lon=121.5,  # , -122.407677
    lat=31.2,  # 37.787937
    buffer_m=2048,
)

# Spatial: bounding box
spatial_bbox = BBox(
    minlon=121.45,
    minlat=31.15,
    maxlon=121.65,
    maxlat=31.25,
)

# Temporal: single year
temporal_year = TemporalSpec.year(2024)

# Temporal: date range
temporal_range = TemporalSpec.range(
    "2024-01-01",
    "2024-06-01",
)

spatial_point, spatial_bbox, temporal_year, temporal_range

## 2. Inspect the raw patch before the model


This is the debugging habit that saves the most time.
If the fetched Sentinel-2 patch already looks wrong, changing the embedding model will not fix it.
The cell below makes the input visible first; the later sections then scale from one ROI to many ROIs.


In [ ]:
check_out = inspect_provider_patch(
    spatial=spatial_point,
    temporal=temporal_range,
    sensor=SensorSpec(
        collection="COPERNICUS/S2_SR_HARMONIZED",
        bands=("B4", "B3", "B2"),
        scale_m=10,
        cloudy_pct=10,
    ),
    return_array=True,
)

show_input_chw(
        check_out["array_chw"],
        title="Sentinel-2 input quicklook",
        rgb_idx=(0, 1, 2),
    )


## 3. One ROI, one model
This is the smallest complete inference loop: same ROI, one model call, one returned embedding.
Once this feels clear, the rest of the package is mostly about changing one axis at a time.


### 3.1 Minimal usage case

In practice, the core shape is:

`(model, spatial, temporal, output) -> Embedding`

Why this is useful:
- you keep the calling pattern stable while switching models
- the same request logic can move from exploration to production export
- output shape stays explicit instead of being hidden in model-specific wrappers



In [ ]:
emb = get_embedding(
    "gse",
    spatial=spatial_point,
    temporal=temporal_year,
    output=OutputSpec.grid(),
)

# print("data.shape:", emb.data.shape)
# print("source:", emb.meta.get("source"))

pca = plot_embedding_pseudocolor(
    emb,
    title="gse (AlphaEarth) PCA pseudocolor",
)

### 3.2 Change preprocessing without changing the rest of the request
Different models impose different input-size constraints. `rs-embed` keeps that complexity in one place through `input_prep`.

- `resize`: fastest and the default option; good for quick comparison and large runs
- `tile`: preserves more spatial detail by splitting the ROI, but can get expensive quickly

<img src="https://raw.githubusercontent.com/cybergis/rs-embed/main/docs/assets/tile_playground.png" width="800" /> 

The point is not only that both modes exist. The point is that you can swap preprocessing while keeping the rest of the API unchanged.

For the full spec surface, see [API specs](https://cybergis.github.io/rs-embed/api_specs/).


In [ ]:
# from rs_embed import InputPrepSpec
emb_resize = get_embedding(
    "terramind",  # 
    spatial=spatial_point,
    temporal=temporal_range,
    output=OutputSpec.grid(),
    input_prep="resize",
)

emb_tile = get_embedding(
    "terramind",  # 
    spatial=spatial_point,
    temporal=temporal_range,
    output=OutputSpec.grid(),
    input_prep="tile",
)

pca = plot_embedding_pseudocolor(
    emb=emb_resize,
    emb2=emb_tile,
    title="resize v.s tile (terramind) PCA pseudocolor",
)

### 3.3 Choose the output shape explicitly

`pooled()` and `grid()` answer different downstream questions.

- `OutputSpec.pooled()`: one vector for the whole ROI; useful for retrieval, classification, and similarity
- `OutputSpec.grid()`: a spatial embedding field; useful when internal structure inside the ROI matters




<img src="https://raw.githubusercontent.com/cybergis/rs-embed/main/docs/assets/output.png" width="800" /> 

Use `pooled` when you want one representation per ROI. Use `grid` when you want structure inside the ROI.


In [ ]:
emb_pooled = get_embedding(
    "satmae",
    spatial=spatial_point,
    temporal=temporal_range,
    output=OutputSpec.pooled(),
)

emb_grid = get_embedding(
    "satmae",
    spatial=spatial_point,
    temporal=temporal_range,
    output=OutputSpec.grid(),
)

print("pooled shape:", emb_pooled.data.shape)
print("grid shape:", emb_grid.data.shape)

## 4. Many ROIs, one model
When the model is fixed and only the locations change, move to `get_embeddings_batch(...)`.
This section keeps the same mental model as before and only scales the `spatial` input from one ROI to many.


In [ ]:
from rs_embed import get_embeddings_batch

points = [
    PointBuffer(lon=121.5, lat=31.2, buffer_m=100),
    PointBuffer(lon=121.6, lat=31.3, buffer_m=100),
    PointBuffer(lon=120.0, lat=30.0, buffer_m=100),
]

embeddings = get_embeddings_batch(
    "satmae", 
    spatials=points,
    temporal=temporal_range, 
    output=OutputSpec.grid(), 
    backend="gee"
)

for i, emb in enumerate(embeddings):
    print(f"Embedding {i} shape: {emb.data.shape}")

## 5. Many ROI x Many Model
`export_batch(...)` is the transition from interactive exploration to repeatable data production.

- export raw inputs and embeddings together
- keep a manifest so every file stays traceable
- scale the same request pattern to multiple ROIs and multiple models

For more detail, see the [export docs](https://cybergis.github.io/rs-embed/api_export/).


In [ ]:
from rs_embed import ExportConfig, ExportTarget, ExportModelRequest

manifest = export_batch(
    spatials=[
        PointBuffer(lon=121.47, lat=31.23, buffer_m=2500),
        PointBuffer(lon=121.47, lat=31.24, buffer_m=2500),
    ],
    temporal=temporal_range,
    models=[
        ExportModelRequest(name="remoteclip"),
        ExportModelRequest(name="terrafm", modality="s1"),
    ],  # or you can spcify as ["remoteclip", "terrafm"]
    target=ExportTarget.per_item("exports", names=["p1", "p2"]),
    output=OutputSpec.pooled(),
    config=ExportConfig(save_inputs=True),
    backend="auto",
)

After export, you usually want one quick sanity check: open a file, confirm shapes and metadata, and visualize the saved input patch.


In [ ]:
manifest, z = inspect_export_npz("exports/p1.npz")